# Data Quality Analysis for Assignment 2

This notebook is a compact quality-focused deliverable for the current offline demo path. It works with local artifacts when they exist and falls back to a small demo-aligned dataframe only when needed. It does **not** represent a production-scale corpus.

## Input Dataset Overview

The goal of this section is to inspect the current offline review data before any cleaning. The notebook prefers local artifacts from the demo path and keeps the fallback explicit so the analysis stays honest and reproducible offline.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import math
import re

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("ggplot")

NOTEBOOK_ROOT = Path.cwd().resolve()
CANDIDATE_PATHS = [
    NOTEBOOK_ROOT / "data" / "interim" / "review_queue_corrected.csv",
    NOTEBOOK_ROOT / "data" / "interim" / "review_queue.csv",
    NOTEBOOK_ROOT / "tests" / "fixtures" / "sample_review_queue_corrected.csv",
    NOTEBOOK_ROOT / "tests" / "fixtures" / "sample_review_queue.csv",
]
STOPWORDS = {
    "the", "and", "for", "with", "that", "this", "from", "have", "has", "was",
    "were", "are", "you", "your", "our", "but", "not", "too", "very", "about",
    "review", "reviews", "supplement", "supplements", "fitness", "product", "products",
    "good", "bad", "great", "nice", "one", "use", "using", "made", "make", "can",
    "would", "could", "just", "really", "more", "less", "after", "before", "also",
    "effect", "effects", "energy", "side", "effect", "warning",
}


def load_quality_frame() -> tuple[pd.DataFrame, str]:
    """Load a local offline artifact or fall back to a small demo-aligned dataframe."""

    for path in CANDIDATE_PATHS:
        if not path.exists():
            continue
        try:
            frame = pd.read_csv(path)
            return frame, f"local artifact: {path.relative_to(NOTEBOOK_ROOT)}"
        except Exception:
            continue

    demo_rows = [
        {"id": "1", "source": "offline_demo", "text": "Great energy boost for workouts and long runs", "label": None, "rating": 5, "created_at": "2026-03-27", "split": None, "meta_json": "{}", "sentiment_label": "positive", "effect_label": "energy", "confidence": 0.96, "reviewed_effect_label": "energy", "review_comment": "", "human_verified": True},
        {"id": "2", "source": "offline_demo", "text": "  Great energy boost for workouts and long runs  ", "label": None, "rating": 5, "created_at": "2026-03-27", "split": None, "meta_json": "{}", "sentiment_label": "positive", "effect_label": "energy", "confidence": 0.92, "reviewed_effect_label": "energy", "review_comment": "duplicate phrasing", "human_verified": True},
        {"id": "3", "source": "offline_demo", "text": "Side effect warning after taking the powder at night", "label": None, "rating": 1, "created_at": "2026-03-27", "split": None, "meta_json": "{}", "sentiment_label": "negative", "effect_label": "side_effects", "confidence": 0.81, "reviewed_effect_label": "side_effects", "review_comment": "", "human_verified": True},
        {"id": "4", "source": "offline_demo", "text": "", "label": None, "rating": 3, "created_at": "2026-03-27", "split": None, "meta_json": "{}", "sentiment_label": "neutral", "effect_label": "other", "confidence": 0.58, "reviewed_effect_label": "other", "review_comment": "empty text example", "human_verified": False},
        {"id": "5", "source": "offline_demo", "text": "Neutral supplement routine with no strong effect", "label": None, "rating": 3, "created_at": "2026-03-27", "split": None, "meta_json": "{}", "sentiment_label": "neutral", "effect_label": "other", "confidence": 0.66, "reviewed_effect_label": "other", "review_comment": "", "human_verified": False},
        {"id": "6", "source": "offline_demo", "text": "Short", "label": None, "rating": 3, "created_at": "2026-03-27", "split": None, "meta_json": "{}", "sentiment_label": "neutral", "effect_label": "other", "confidence": 0.55, "reviewed_effect_label": "other", "review_comment": "too short to be useful", "human_verified": False},
        {"id": "7", "source": "offline_demo", "text": "Energy and focus felt stronger during training sessions", "label": None, "rating": 5, "created_at": "2026-03-27", "split": None, "meta_json": "{}", "sentiment_label": "positive", "effect_label": "energy", "confidence": 0.91, "reviewed_effect_label": "energy", "review_comment": "", "human_verified": True},
        {"id": "8", "source": "offline_demo", "text": "A bit of stomach discomfort after the first dose", "label": None, "rating": 2, "created_at": "2026-03-27", "split": None, "meta_json": "{}", "sentiment_label": "negative", "effect_label": "side_effects", "confidence": 0.74, "reviewed_effect_label": "side_effects", "review_comment": "", "human_verified": False},
        {"id": "9", "source": "offline_demo", "text": "Balanced routine with mixed results and no clear side effects", "label": None, "rating": 3, "created_at": "2026-03-27", "split": None, "meta_json": "{}", "sentiment_label": "neutral", "effect_label": "other", "confidence": 0.61, "reviewed_effect_label": "other", "review_comment": "", "human_verified": False},
        {"id": "10", "source": "offline_demo", "text": "Energy boost but the label text repeats energy energy energy too much", "label": None, "rating": 4, "created_at": "2026-03-27", "split": None, "meta_json": "{}", "sentiment_label": "positive", "effect_label": "energy", "confidence": 0.88, "reviewed_effect_label": "energy", "review_comment": "long repeated text", "human_verified": True},
    ]
    return pd.DataFrame(demo_rows), "inline demo-aligned dataframe"


def canonical_text_column(frame: pd.DataFrame) -> str:
    """Pick the text column used by the current text -> effect_label task."""

    for candidate in ("text", "content", "review_text", "body"):
        if candidate in frame.columns:
            return candidate
    return frame.columns[0]


def baseline_label_column(frame: pd.DataFrame) -> str | None:
    """Pick the baseline label source for the current offline analysis.

    The baseline analysis prefers effect_label because it matches the current ML task.
    reviewed_effect_label is a separate human-reviewed correction layer and is not treated
    as the primary source for the baseline label distribution.
    """

    for candidate in ("effect_label", "label", "sentiment_label"):
        if candidate in frame.columns and frame[candidate].notna().any():
            return candidate
    return None


def normalize_whitespace(text: object) -> str:
    """Normalize whitespace while keeping the original meaning intact."""

    if text is None:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def normalize_text_key(text: object) -> str:
    """Normalize a text value for duplicate detection in the notebook."""

    return normalize_whitespace(text).lower()


def tokenize(text: str) -> list[str]:
    """Simple regex-based tokenization for EDA and cleaning diagnostics."""

    tokens = re.findall(r"[a-zA-Z']+", text.lower())
    return [token for token in tokens if len(token) > 2 and token not in STOPWORDS]


def iqr_bounds(values: pd.Series) -> tuple[float, float]:
    """Compute robust IQR bounds for a numeric series."""

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    return float(q1 - 1.5 * iqr), float(q3 + 1.5 * iqr)


def normalized_duplicate_count(frame: pd.DataFrame, text_column: str) -> int:
    """Count duplicates after normalizing text to a stable comparison key."""

    normalized = frame[text_column].fillna("").map(normalize_text_key)
    return int(normalized.duplicated().sum())


def row_metrics(frame: pd.DataFrame, text_column: str, label_column: str | None) -> dict[str, object]:
    """Summarize the most important quality metrics for before/after comparison."""

    text_series = frame[text_column].fillna("").astype(str)
    word_counts = text_series.map(lambda value: len(tokenize(value)))
    label_series = frame[label_column].fillna("missing").astype(str).str.strip() if label_column else pd.Series([], dtype="object")
    label_counts = label_series.replace("", "missing").value_counts().to_dict() if label_column else {}
    empty_text = int((text_series.str.strip() == "").sum())
    duplicates = normalized_duplicate_count(frame, text_column)
    return {
        "n_rows": int(len(frame)),
        "duplicates": duplicates,
        "empty_text": empty_text,
        "mean_words": float(word_counts.mean()) if len(word_counts) else 0.0,
        "min_words": int(word_counts.min()) if len(word_counts) else 0,
        "max_words": int(word_counts.max()) if len(word_counts) else 0,
        "label_counts": label_counts,
    }


df, source_note = load_quality_frame()
text_column = canonical_text_column(df)
label_column = baseline_label_column(df)
reviewed_label_column = "reviewed_effect_label" if "reviewed_effect_label" in df.columns else None

print(f"Data source: {source_note}")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
display(df.head(5))

## Detected Quality Issues

The following checks mirror the same practical concerns used in the pipeline: missing values, duplicates, text-length outliers, and class imbalance. The baseline label distribution is built from `effect_label` when it is available. `reviewed_effect_label` is treated as a separate human-reviewed correction layer, not the primary baseline label source. The goal is not to overfit the analysis, but to identify issues that matter for the current `text -> effect_label` task.

In [ ]:
base_metrics = row_metrics(df, text_column, label_column)
text_series = df[text_column].fillna("").astype(str)
word_counts = text_series.map(lambda value: len(tokenize(value)))
lower, upper = iqr_bounds(word_counts) if len(word_counts) else (0.0, 0.0)
outlier_mask = (word_counts < lower) | (word_counts > upper)
outliers = int(outlier_mask.sum())

label_for_display = label_column if label_column is not None else reviewed_label_column

issue_summary = pd.DataFrame([
    {"issue": "missing_text", "count": base_metrics["empty_text"]},
    {"issue": "duplicates_after_normalization", "count": base_metrics["duplicates"]},
    {"issue": "word_count_outliers", "count": outliers},
    {"issue": "label_imbalance_note", "count": int(pd.Series(base_metrics["label_counts"]).max() - pd.Series(base_metrics["label_counts"]).min()) if base_metrics["label_counts"] else 0},
])
issue_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(word_counts, bins=min(12, max(1, len(word_counts))))
axes[0].set_title("Word count distribution")
axes[0].set_xlabel("words")
axes[0].set_ylabel("rows")

if label_column and base_metrics["label_counts"]:
    pd.Series(base_metrics["label_counts"]).head(20).plot(kind="bar", ax=axes[1], color="slateblue")
    axes[1].set_title("Label distribution")
    axes[1].set_xlabel("label")
    axes[1].set_ylabel("count")
    plt.xticks(rotation=45, ha="right")
else:
    axes[1].axis("off")

fig.tight_layout()
plt.show()

## Strategy A Cleaning

Strategy A is conservative: normalize whitespace, drop empty text, drop duplicates, and remove IQR outliers in text length. This is a good baseline when we want to preserve the raw semantic content of the remaining rows and avoid altering text that will later feed the `text -> effect_label` model.

In [ ]:
def clean_strategy_a(frame: pd.DataFrame, text_column: str) -> pd.DataFrame:
    """Apply conservative cleaning that removes noisy rows without rewriting text."""

    cleaned = frame.copy()
    cleaned[text_column] = cleaned[text_column].fillna("").map(normalize_whitespace)
    cleaned = cleaned[cleaned[text_column] != ""].copy()
    cleaned = cleaned.drop_duplicates().copy()
    if cleaned.empty:
        return cleaned
    local_word_counts = cleaned[text_column].map(lambda value: len(tokenize(value)))
    low, high = iqr_bounds(local_word_counts)
    return cleaned[(local_word_counts >= low) & (local_word_counts <= high)].copy()

strategy_a = clean_strategy_a(df, text_column)
strategy_a_metrics = row_metrics(strategy_a, text_column, label_column)
strategy_a_metrics

## Strategy B Cleaning

Strategy B is more aggressive: normalize whitespace, drop empty text, drop duplicates, keep only texts above a minimum word threshold, and clip extreme text lengths to an IQR-based bound by truncating tokens. This is a notebook-level proxy for aggressive cleaning, not a precise production transformation. It can keep more rows, but it may also reduce semantic detail in longer reviews.

In [ ]:
def clip_tokens(text: str, max_tokens: int) -> str:
    """Clip text to a token window while keeping the start of the review intact."""

    tokens = tokenize(text)
    if len(tokens) <= max_tokens:
        return text
    return " ".join(tokens[:max_tokens])


def clean_strategy_b(frame: pd.DataFrame, text_column: str) -> pd.DataFrame:
    """Apply a more aggressive cleaning path with a minimum-length filter and clipping."""

    cleaned = frame.copy()
    cleaned[text_column] = cleaned[text_column].fillna("").map(normalize_whitespace)
    cleaned = cleaned[cleaned[text_column] != ""].copy()
    cleaned = cleaned.drop_duplicates().copy()
    if cleaned.empty:
        return cleaned

    cleaned["_word_count"] = cleaned[text_column].map(lambda value: len(tokenize(value)))
    low, high = iqr_bounds(cleaned["_word_count"])
    min_words = max(3, int(math.floor(low)) if low > 0 else 3)
    cleaned = cleaned[cleaned["_word_count"] >= min_words].copy()
    if cleaned.empty:
        return cleaned.drop(columns=["_word_count"], errors="ignore")

    cap = max(min_words, int(math.ceil(high)) if high > 0 else min_words)
    cleaned[text_column] = cleaned[text_column].map(lambda value: clip_tokens(value, cap))
    return cleaned.drop(columns=["_word_count"], errors="ignore")


strategy_b = clean_strategy_b(df, text_column)
strategy_b_metrics = row_metrics(strategy_b, text_column, label_column)
strategy_b_metrics

## Before/After Comparison

The table below compares the original data with the two candidate cleaning strategies. The main checks are row count, duplicates, empty text, and average word count. That is enough to judge whether we are making the dataset cleaner without destroying useful training signal.

In [ ]:
comparison = pd.DataFrame([
    {"stage": "before", **{k: base_metrics[k] for k in ["n_rows", "duplicates", "empty_text", "mean_words"]}},
    {"stage": "strategy_a", **{k: strategy_a_metrics[k] for k in ["n_rows", "duplicates", "empty_text", "mean_words"]}},
    {"stage": "strategy_b", **{k: strategy_b_metrics[k] for k in ["n_rows", "duplicates", "empty_text", "mean_words"]}},
])
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison.set_index("stage")["n_rows"].plot(kind="bar", ax=axes[0], color="teal")
axes[0].set_title("Rows retained by stage")
axes[0].set_xlabel("stage")
axes[0].set_ylabel("n_rows")
comparison.set_index("stage")["mean_words"].plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Average words by stage")
axes[1].set_xlabel("stage")
axes[1].set_ylabel("mean_words")
fig.tight_layout()
plt.show()

## Short Argumentation / Chosen Strategy

For the current `text -> effect_label` task, Strategy A is the better default. It removes empty text, duplicates, and extreme-length outliers without rewriting the text itself, so the model still sees the full semantic content of each retained review.

Strategy B is useful when the dataset has many very short or noisy records, but truncating long reviews can remove label-bearing detail. In this offline demo setting, preserving the original text signal is more valuable than aggressively compressing it.

## Limitations

- The notebook uses the current offline demo path or a small demo-aligned fallback, not a production corpus.
- The data volume is intentionally small, so class imbalance and outlier behavior are only directional signals.
- Strategy B is a notebook-level proxy for a more aggressive cleanup policy; it is not a replacement for a production preprocessing pipeline.
- The argumentation is tailored to the current baseline task and should be revisited if the label schema or text distribution changes.
- `reviewed_effect_label` remains a human-reviewed correction layer and should not be treated as the baseline distribution source.